In [19]:
import os, sys, tempfile
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import mlflow
import ray
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from torch.utils.data import Dataset, DataLoader
import math


In [4]:
MLFLOW_TRACKING_URI = "http://145.38.192.114:5002"   # <-- paste your server URL here
EXPERIMENT_NAME     = "team_transformer_magic"

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
BASE_DIR  = Path("/content") if IN_COLAB else Path(".")
DATA_DIR  = (BASE_DIR / "data").resolve()  # absolute paths matter for Ray
RAY_DIR   = str((BASE_DIR / "ray_results").resolve())

In [5]:
from pathlib import Path

from vectormesh.data.cache import VectorCache

artefacts = Path("../artefacts")

# these globs use the `legal_bert` caches. You can change it to pick the `legal_dutch` caches.
trainpath = next(artefacts.glob("*bert*train/")) 
validpath = next(artefacts.glob("*bert*valid/"))

if not (trainpath.is_dir() and validpath.is_dir()):
    raise FileNotFoundError(f"Data folder not found")

traincache = VectorCache.load(path=trainpath)
validcache = VectorCache.load(path=validpath)

2026-06-08 15:27:10.190 | SUCCESS  | vectormesh.data.cache:load:140 - Cache loaded from ../artefacts/20260113075724_aktes_theshold_50_d97342_legal_bert_train
2026-06-08 15:27:10.260 | SUCCESS  | vectormesh.data.cache:load:140 - Cache loaded from ../artefacts/20260113080216_aktes_theshold_50_d97342_legal_bert_valid


In [6]:
import json
def print_cache_info(cache: VectorCache):
    print(f"Cache name: {cache.name}")
    print(f"Cache directory: {cache.cache_dir}")
    print(f"Dataset: {cache.dataset}")
    print(f"Metadata: {json.dumps(cache.metadata, indent=2)}")

print_cache_info(traincache)

Cache name: 20260113075724_aktes_theshold_50_d97342_legal_bert_train
Cache directory: /Users/saifzaman/Documents/DataScience/transformer-vectormesh/artefacts
Dataset: Dataset({
    features: ['text', 'target', 'labels', 'legal_dutch'],
    num_rows: 15532
})
Metadata: {
  "legal_dutch": {
    "vectormesh_version": "0.1.0",
    "model_tag": "Gerwin/legal-bert-dutch-english",
    "vectorizer_type": "Vectorizer",
    "tensordtype": 2,
    "hidden_size": 768,
    "context_size": 512,
    "chunk_sizes": {}
  },
  "features": [
    "text",
    "target",
    "labels",
    "legal_dutch"
  ],
  "created_at": "2026-01-13T07:57:34.682329",
  "num_observations": 15532
}


In [7]:
print_cache_info(validcache)

Cache name: 20260113080216_aktes_theshold_50_d97342_legal_bert_valid
Cache directory: /Users/saifzaman/Documents/DataScience/transformer-vectormesh/artefacts
Dataset: Dataset({
    features: ['text', 'target', 'labels', 'legal_dutch'],
    num_rows: 1942
})
Metadata: {
  "legal_dutch": {
    "vectormesh_version": "0.1.0",
    "model_tag": "Gerwin/legal-bert-dutch-english",
    "vectorizer_type": "Vectorizer",
    "tensordtype": 2,
    "hidden_size": 768,
    "context_size": 512,
    "chunk_sizes": {
      "8": 113,
      "13": 90,
      "4": 68,
      "19": 39,
      "10": 179,
      "15": 94,
      "11": 153,
      "14": 88,
      "9": 156,
      "3": 119,
      "27": 14,
      "2": 91,
      "12": 130,
      "60": 1,
      "6": 41,
      "23": 31,
      "22": 21,
      "5": 51,
      "7": 68,
      "51": 3,
      "26": 18,
      "21": 28,
      "29": 9,
      "35": 10,
      "18": 32,
      "32": 4,
      "38": 3,
      "24": 17,
      "16": 73,
      "17": 55,
      "30": 7,
      "

In [8]:
train = traincache
valid = validcache

In [9]:
column_name = "legal_dutch"  

In [12]:
# --- 1. Dataset Wrapper ---
class CachedVectorDataset(Dataset):
    """Wraps the VectorCache into a PyTorch-compatible Dataset."""
    def __init__(self, cache, vector_key="legal_dutch", label_key="labels"):
        self.cache = cache
        self.vector_key = vector_key
        self.label_key = label_key
        
    def __len__(self):
        return len(self.cache)

    def __getitem__(self, idx):
        item = self.cache[idx] 
        return {
            # Standardizes the key names for the collate function
            "embeddings": item[self.vector_key],
            "labels": item[self.label_key]
        }

In [13]:
# --- 2. Collate Function ---
NUM_CLASSES = 32 

def padded_document_collate_fn(batch):
    embeddings_list = []
    labels_list = []
    
    for item in batch:
        # Note: We now look for "embeddings", not "legal_dutch"
        emb = torch.tensor(item["embeddings"], dtype=torch.float32)
        if emb.dim() == 1:
            emb = emb.view(-1, 768) 
        embeddings_list.append(emb)
        
        # Multi-hot encode the labels
        multi_hot_labels = torch.zeros(NUM_CLASSES, dtype=torch.float32)
        active_indices = item["labels"]
        
        if torch.is_tensor(active_indices):
            active_indices = active_indices.tolist()
        if isinstance(active_indices, int):
            active_indices = [active_indices]
            
        if len(active_indices) > 0:
            multi_hot_labels[active_indices] = 1.0
            
        labels_list.append(multi_hot_labels)
    
    # Pad sequences and create masks
    padded_embeddings = nn.utils.rnn.pad_sequence(embeddings_list, batch_first=True)
    lengths = torch.tensor([emb.size(0) for emb in embeddings_list])
    max_len = padded_embeddings.size(1)
    
    padding_mask = torch.arange(max_len).expand(len(lengths), max_len) >= lengths.unsqueeze(1)
    labels = torch.stack(labels_list)
    
    return padded_embeddings, padding_mask, labels

In [14]:
# --- 3. DataLoader Setup ---
# Initialize the datasets (make sure your traincache and validcache are loaded)
train_dataset = CachedVectorDataset(cache=traincache, vector_key="legal_dutch")
valid_dataset = CachedVectorDataset(cache=validcache, vector_key="legal_dutch")

BATCH_SIZE = 16

# Important: make sure collate_fn points to padded_document_collate_fn
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=padded_document_collate_fn
)

valid_loader = DataLoader(
    valid_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=padded_document_collate_fn
)

In [17]:


class PositionalEncoding(nn.Module):
    """Adds positional information to the precomputed chunks."""
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(1, max_len, d_model) # Batch first
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: [batch_size, num_chunks, embedding_dim]
        return x + self.pe[:, :x.size(1), :]

class LabelWiseAttention(nn.Module):
    """32 per-class attentions -> 32 logits"""
    def __init__(self, hidden_size: int, num_classes: int):
        super().__init__()
        self.attention_weights = nn.Linear(hidden_size, num_classes)
        self.class_weights = nn.Parameter(torch.randn(num_classes, hidden_size))
        self.class_bias = nn.Parameter(torch.zeros(num_classes))
        
        nn.init.xavier_uniform_(self.attention_weights.weight)
        nn.init.xavier_uniform_(self.class_weights)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        # 1. Attention Scores
        A = self.attention_weights(x) 
        
        if mask is not None:
            A = A.masked_fill(mask.unsqueeze(-1), float('-inf'))
            
        A = torch.softmax(A, dim=1) 
        
        # 2. Class-specific context vectors
        V = torch.bmm(A.transpose(1, 2), x)
        
        # 3. Compute independent logits per class
        logits = (V * self.class_weights).sum(dim=-1) + self.class_bias 
        
        return logits

class ChunkAttentionClassifier(nn.Module):
    """Full architecture based on the diagram."""
    def __init__(self, hidden_size=768, num_classes=32, num_layers=2, nhead=8):
        super().__init__()
        self.pos_encoder = PositionalEncoding(hidden_size)
        
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=hidden_size, 
            nhead=nhead, 
            dim_feedforward=hidden_size * 4,
            batch_first=True,
            activation='gelu'
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.label_attention = LabelWiseAttention(hidden_size, num_classes)
        
    def forward(self, x: torch.Tensor, padding_mask: torch.Tensor = None):
        x = self.pos_encoder(x)
        contextualized = self.transformer_encoder(x, src_key_padding_mask=padding_mask)
        logits = self.label_attention(contextualized, mask=padding_mask)
        return logits

In [24]:
config = {
    "epochs": 3,
    "batch_size": 16,
    "learning_rate": 1e-4,
    # "hidden_size": 768,
    "hidden_size": 128,
    "num_classes": 32,
    "num_layers": 3,
    "threshold": 0.5
}

In [25]:
PRINT_EVERY = 10

# Ensure model and optimizer are initialized
model = ChunkAttentionClassifier(
    hidden_size=config["hidden_size"], 
    num_classes=config["num_classes"], 
    num_layers=config["num_layers"]
)
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])
criterion = nn.BCEWithLogitsLoss()

# --- Setup MLflow ---
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print("Starting training loop...")

with mlflow.start_run():
    # Log your configuration and tags
    mlflow.log_params(config)
    mlflow.set_tag("framework", "pytorch")
    mlflow.set_tag("search", "team-transformer")
    
    global_step = 0 # Track total batches across all epochs for MLflow
    
    for epoch in range(1, config["epochs"] + 1):
        print(f"\n{'='*10} Epoch {epoch}/{config['epochs']} {'='*10}")
        
        # --- TRAINING PHASE ---
        model.train()
        running_loss = 0.0
        running_acc = 0.0
        
        for iteration, (batch_embs, batch_mask, batch_labels) in enumerate(train_loader, start=1):
            global_step += 1
            optimizer.zero_grad()
            
            # Forward pass
            logits = model(batch_embs, padding_mask=batch_mask)
            loss = criterion(logits, batch_labels)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            # Calculate Multi-Label Accuracy
            with torch.no_grad():
                probs = torch.sigmoid(logits)
                preds = (probs >= config["threshold"]).float()
                batch_acc = (preds == batch_labels).float().mean().item()
                
            running_loss += loss.item()
            running_acc += batch_acc
            
            # Log batch-level metrics to MLflow
            mlflow.log_metric("train_loss_step", loss.item(), step=global_step)
            mlflow.log_metric("train_acc_step", batch_acc, step=global_step)
            
            # Print progress
            if iteration % PRINT_EVERY == 0 or iteration == 1:
                print(f"Iter: {iteration:4d} | Train Loss: {loss.item():.4f} | Train Acc: {batch_acc:.4f}")

        # --- VALIDATION PHASE (End of Epoch) ---
        model.eval()
        val_loss = 0.0
        val_acc = 0.0
        
        with torch.no_grad():
            for val_iter, (val_embs, val_mask, val_labels) in enumerate(valid_loader, start=1):
                val_logits = model(val_embs, padding_mask=val_mask)
                v_loss = criterion(val_logits, val_labels)
                
                val_probs = torch.sigmoid(val_logits)
                val_preds = (val_probs >= config["threshold"]).float()
                v_acc = (val_preds == val_labels).float().mean().item()
                
                val_loss += v_loss.item()
                val_acc += v_acc
                
        # Calculate averages for the epoch
        avg_train_loss = running_loss / len(train_loader)
        avg_train_acc = running_acc / len(train_loader)
        avg_val_loss = val_loss / len(valid_loader)
        avg_val_acc = val_acc / len(valid_loader)
        
        # Log epoch-level metrics to MLflow
        mlflow.log_metric("train_loss_epoch", avg_train_loss, step=epoch)
        mlflow.log_metric("train_acc_epoch", avg_train_acc, step=epoch)
        mlflow.log_metric("val_loss_epoch", avg_val_loss, step=epoch)
        mlflow.log_metric("val_acc_epoch", avg_val_acc, step=epoch)
        
        print("-" * 40)
        print(f"Epoch {epoch} Summary:")
        print(f"Avg Train Loss: {avg_train_loss:.4f} | Avg Train Acc: {avg_train_acc:.4f}")
        print(f"Avg Val Loss:   {avg_val_loss:.4f} | Avg Val Acc:   {avg_val_acc:.4f}")
        print("-" * 40)
        
    print("Training complete! Run 'mlflow ui' in your terminal to view the dashboard.")

Starting training loop...

========== Epoch 1/3 ==========
🏃 View run clean-penguin-553 at: http://145.38.192.114:5002/#/experiments/1/runs/4a4090f305934c3496155d5ba27c1846
🧪 View experiment at: http://145.38.192.114:5002/#/experiments/1


/var/folders/_2/dxlhmf4n7cxfcfk2zncyr3lw0000gp/T/ipykernel_16261/630224672.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  emb = torch.tensor(item["embeddings"], dtype=torch.float32)


RuntimeError: The size of tensor a (768) must match the size of tensor b (128) at non-singleton dimension 2